# Mini-World GPT — train from scratch + visualize internals

This notebook trains a **small Raschka-style GPT** on the controlled `mini-world.txt` corpus, saves the trained weights, traces one real forward pass, and opens the LLM debugger.

Why this version is better for learning than the original `the-verdict.txt` example:

- the corpus has only about 100 important words;
- concepts repeat consistently (`cat -> fish`, `dog -> meat`, `car -> road`, etc.);
- we use a tiny corpus tokenizer instead of the full 50,257-token GPT-2 vocabulary;
- the model is therefore genuinely small and much easier to train and inspect.

Pipeline:

`mini-world.txt -> tiny tokenizer -> small GPT -> checkpoint -> trace.json -> debugger.html`


## 1. Install dependency and clone the code repositories

In [ ]:
!pip -q install tiktoken

import os
import sys
import glob
import json
import base64
import subprocess
from pathlib import Path

WORKING = "/kaggle/working"
RASCHKA_DIR = f"{WORKING}/LLMs-from-scratch"
VIS_DIR = f"{WORKING}/llm-visualize"

def clone_once(url, target):
    if not os.path.exists(target):
        subprocess.run(["git", "clone", "--depth", "1", url, target], check=True)
    else:
        print(f"Already exists: {target}")

clone_once("https://github.com/rasbt/LLMs-from-scratch.git", RASCHKA_DIR)
clone_once("https://github.com/gerasidev/llm-visualize.git", VIS_DIR)

CH05_DIR = f"{RASCHKA_DIR}/ch05/01_main-chapter-code"
sys.path.insert(0, CH05_DIR)
sys.path.insert(0, VIS_DIR)

print("Raschka code:", CH05_DIR)
print("Visualizer:", VIS_DIR)

## 2. Load `mini-world.txt`

The dataset repository is private. This cell tries, in order:

1. a `mini-world.txt` file already attached under `/kaggle/input`;
2. the private GitHub repository using a Kaggle secret called `GITHUB_TOKEN`.

If you attach the dataset to the notebook as a Kaggle input, you do not need a GitHub token.

In [ ]:
import requests

def load_mini_world():
    candidates = glob.glob("/kaggle/input/**/mini-world.txt", recursive=True)
    if candidates:
        path = candidates[0]
        print("Loading Kaggle input:", path)
        return Path(path).read_text(encoding="utf-8"), path

    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass

    if token:
        api_url = "https://api.github.com/repos/gerasidev/mini-world-dataset/contents/data/mini-world.txt"
        response = requests.get(
            api_url,
            headers={
                "Authorization": f"Bearer {token}",
                "Accept": "application/vnd.github+json",
            },
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        text = base64.b64decode(payload["content"]).decode("utf-8")
        print("Loaded private GitHub dataset.")
        return text, api_url

    raise FileNotFoundError(
        "mini-world.txt was not found. Either attach it as a Kaggle input "
        "or add a Kaggle secret named GITHUB_TOKEN with access to "
        "gerasidev/mini-world-dataset."
    )

text_data, data_source = load_mini_world()

print("Characters:", len(text_data))
print("Whitespace words:", len(text_data.split()))
print("Source:", data_source)
print()
print(text_data[:500])

## 3. Build a tiny tokenizer from this corpus

For this learning model we deliberately **do not use the GPT-2 50,257-token vocabulary**.

The tokenizer separates lowercase words and punctuation, then gives each unique token an integer ID. This keeps the vocabulary close to the actual mini-world vocabulary.

In [ ]:
import re
from collections import Counter

TOKEN_PATTERN = re.compile(r"[a-z]+|[.,!?;:]")

class MiniWorldTokenizer:
    def __init__(self, text=None, stoi=None):
        if stoi is None:
            tokens = TOKEN_PATTERN.findall(text.lower())
            vocab = ["<unk>"] + sorted(set(tokens))
            self.stoi = {token: i for i, token in enumerate(vocab)}
        else:
            self.stoi = dict(stoi)
        self.itos = {i: token for token, i in self.stoi.items()}

    def encode(self, text):
        tokens = TOKEN_PATTERN.findall(text.lower())
        unk = self.stoi["<unk>"]
        return [self.stoi.get(token, unk) for token in tokens]

    def decode(self, ids):
        tokens = [self.itos[int(i)] for i in ids]
        out = ""
        for token in tokens:
            if token in ".,!?;:":
                out = out.rstrip() + token + " "
            else:
                out += token + " "
        return out.strip()

tokenizer = MiniWorldTokenizer(text_data)
all_ids = tokenizer.encode(text_data)

print("Vocabulary size:", len(tokenizer.stoi))
print("Corpus tokens:", len(all_ids))
print("First 30 vocabulary entries:")
print(list(tokenizer.stoi.items())[:30])
print()
print("Example:", tokenizer.decode(tokenizer.encode("john has a cat. the cat eats fish.")))

## 4. Create next-token training data

Exactly like GPT training, the target is the input shifted one token to the left.

Example:

`input:  john has a cat`  
`target: has a cat .`

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TokenDataset(Dataset):
    def __init__(self, token_ids, max_length, stride):
        self.inputs = []
        self.targets = []
        for i in range(0, len(token_ids) - max_length, stride):
            x = token_ids[i:i + max_length]
            y = token_ids[i + 1:i + max_length + 1]
            if len(y) == max_length:
                self.inputs.append(torch.tensor(x, dtype=torch.long))
                self.targets.append(torch.tensor(y, dtype=torch.long))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

CONTEXT_LENGTH = 32
train_split = int(0.90 * len(all_ids))
train_ids = all_ids[:train_split]
val_ids = all_ids[train_split:]

train_dataset = TokenDataset(train_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)
val_dataset = TokenDataset(val_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, drop_last=False)

x, y = train_dataset[0]
print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))
print("Input :", tokenizer.decode(x.tolist()))
print("Target:", tokenizer.decode(y.tolist()))

## 5. Create a genuinely small GPT

We keep Raschka's GPT architecture but reduce its dimensions for this controlled corpus.

This is intentional: we want a model whose internals are easy to train and inspect, not a 124M-parameter language model.

In [ ]:
from previous_chapters import GPTModel, generate_text_simple
from gpt_train import train_model_simple, calc_loss_loader

from python.trace_raschka import (
    save_model_checkpoint,
    load_model_checkpoint,
    trace_forward_pass,
    write_trace,
)
from python.export_visualizer import build_visualizer

SMALL_GPT_CONFIG = {
    "vocab_size": len(tokenizer.stoi),
    "context_length": CONTEXT_LENGTH,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

TRAIN_SETTINGS = {
    "learning_rate": 5e-4,
    "num_epochs": 12,
    "weight_decay": 0.1,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)

model = GPTModel(SMALL_GPT_CONFIG).to(device)
parameter_count = sum(p.numel() for p in model.parameters())

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Vocabulary:", SMALL_GPT_CONFIG["vocab_size"])
print("Parameters:", f"{parameter_count:,}")
print("Layers:", SMALL_GPT_CONFIG["n_layers"])
print("Heads:", SMALL_GPT_CONFIG["n_heads"])
print("Embedding dimension:", SMALL_GPT_CONFIG["emb_dim"])

## 6. Train once, or load a checkpoint

For the first run leave `CHECKPOINT_TO_LOAD = None`.

After training, save the Kaggle notebook version so `mini-world-gpt.pth` becomes an output. In a later session attach that output and set `CHECKPOINT_TO_LOAD` to its path. Then this notebook skips training.

In [ ]:
# Example for a future Kaggle session:
# CHECKPOINT_TO_LOAD = "/kaggle/input/<your-saved-output>/mini-world-gpt.pth"
CHECKPOINT_TO_LOAD = None

CHECKPOINT_OUT = "/kaggle/working/mini-world-gpt.pth"
TRACE_OUT = "/kaggle/working/mini-world-trace.json"
HTML_OUT = "/kaggle/working/mini-world-debugger.html"

if CHECKPOINT_TO_LOAD:
    load_model_checkpoint(model, CHECKPOINT_TO_LOAD, device=device)
    print("Loaded:", CHECKPOINT_TO_LOAD)
else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_SETTINGS["learning_rate"],
        weight_decay=TRAIN_SETTINGS["weight_decay"],
    )

    train_losses, val_losses, tokens_seen = train_model_simple(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=TRAIN_SETTINGS["num_epochs"],
        eval_freq=20,
        eval_iter=5,
        start_context="john has a cat",
        tokenizer=tokenizer,
    )

    save_model_checkpoint(
        model,
        CHECKPOINT_OUT,
        config=SMALL_GPT_CONFIG,
        optimizer=optimizer,
        extra={
            "dataset": "mini-world.txt",
            "epochs": TRAIN_SETTINGS["num_epochs"],
            "tokenizer_vocab": tokenizer.stoi,
        },
    )
    print("Saved checkpoint:", CHECKPOINT_OUT)

## 7. Check whether the tiny model learned the mini-world patterns

In [ ]:
model.eval()

def generate(prompt, max_new_tokens=20):
    ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        out = generate_text_simple(
            model=model,
            idx=ids,
            max_new_tokens=max_new_tokens,
            context_size=SMALL_GPT_CONFIG["context_length"],
        )
    return tokenizer.decode(out[0].tolist())

for prompt in [
    "john has a cat",
    "the cat eats",
    "mary has a dog",
    "the car moves on the",
    "the horse lives on the",
]:
    print("PROMPT:", prompt)
    print(generate(prompt, 12))
    print("-")

## 8. Optional: inspect learned embedding similarity

This is a simple first sanity check before the full debugger. It compares the learned embedding vectors using cosine similarity.

In [ ]:
import torch.nn.functional as F

def embedding_similarity(word_a, word_b):
    a = model.tok_emb.weight[tokenizer.stoi[word_a]].detach()
    b = model.tok_emb.weight[tokenizer.stoi[word_b]].detach()
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

pairs = [
    ("cat", "dog"),
    ("cat", "bird"),
    ("car", "bus"),
    ("car", "truck"),
    ("cat", "car"),
    ("fish", "meat"),
]

for a, b in pairs:
    print(f"{a:>6} vs {b:<6}: {embedding_similarity(a, b): .4f}")

## 9. Trace one forward pass

Now we record the real internal tensors for one short prompt. This does **not** train or modify the model.

In [ ]:
TRACE_TEXT = "john has a cat. the cat eats"

trace_ids = tokenizer.encode(TRACE_TEXT)
input_ids = torch.tensor(trace_ids, dtype=torch.long).unsqueeze(0).to(device)

trace = trace_forward_pass(
    model,
    input_ids,
    tokens=[tokenizer.itos[i] for i in trace_ids],
    decode_token=lambda i: tokenizer.itos[int(i)],
    top_k=10,
)

write_trace(trace, TRACE_OUT)

print("Trace text:", TRACE_TEXT)
print("Trace tokens:", [(t["text"], t["id"]) for t in trace["tokens"]])
print("Layers traced:", len(trace["layers"]))
print("Saved:", TRACE_OUT)

## 10. Build the standalone visual debugger

In [ ]:
build_visualizer(
    trace_path=TRACE_OUT,
    output_path=HTML_OUT,
)

print("Debugger saved:", HTML_OUT)

## 11. Display the debugger in Kaggle

If Kaggle sanitizes JavaScript in the inline notebook renderer, open/download `mini-world-debugger.html` from the notebook output files.

In [ ]:
from IPython.display import HTML, display, FileLink

with open(HTML_OUT, "r", encoding="utf-8") as f:
    display(HTML(f.read()))

print("Files to keep:")
if os.path.exists(CHECKPOINT_OUT):
    display(FileLink(CHECKPOINT_OUT))
display(FileLink(TRACE_OUT))
display(FileLink(HTML_OUT))